# 🔬 Entrenamiento de Alta Precisión YOLO11 - Bromatología UTEQ
### Proyecto: Asistente Inteligente de Laboratorio con Visión Artificial y RAG
**Universidad Técnica Estatal de Quevedo (UTEQ)**  
**Facultad de Ciencias Pecuarias y Biológicas**

---

Este cuaderno entrena **YOLO11 Nano (`yolo11n.pt`)** con un **dataset balanceado y cajas delimitadoras precisas** para 20 equipos del laboratorio, garantizando recuadros ceñidos y máxima exactitud en la detección.

## 🛠️ Paso 1: Configurar GPU y Dependencias
Verifica que estés conectado con **GPU T4** en el menú superior (`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4`).

In [ ]:
# Verificar GPU
!nvidia-smi

# Instalar Ultralytics y dependencias
!pip install -q --upgrade ultralytics tensorflow albumentations opencv-python matplotlib

import ultralytics
ultralytics.checks()

## 📦 Paso 2: Cargar y Descomprimir el Dataset de Alta Precisión (`dataset_bromatologia_preciso.zip`)
Sube el archivo `dataset_bromatologia_preciso.zip` (25 MB - se sube en 5 segundos).

In [ ]:
import os
import glob
import zipfile
from google.colab import files

# Limpiar dataset anterior si existe
!rm -rf dataset runs /content/dataset*

print("Por favor selecciona el archivo 'dataset_bromatologia_preciso.zip' de tu computadora:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

print(f"Descomprimiendo {zip_name}...")
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('.')

print("✅ Dataset de Alta Precisión descomprimido exitosamente:")
!ls -lh dataset/images/train | head -n 10

## ⚙️ Paso 3: Configurar `data.yaml` con las 20 Clases

In [ ]:
data_yaml_content = """
path: /content/dataset
train: images/train
val: images/val
test: images/test

nc: 20
names:
  0: destilador_kjeldahl
  1: analizador_fibra
  2: placa_calefactora_heidolph
  3: phmetro_ohaus
  4: molino_ciclonico_foss
  5: estufa_secado_memmert
  6: refractometro_atago
  7: calorimetro_bomba
  8: campana_extraccion_gases
  9: cabina_flujo_laminar_uvp
  10: sistema_tratamiento_agua
  11: destilador_agua
  12: bomba_vacio_recirculacion
  13: bomba_vacio_membrana
  14: agitador_vortex
  15: gradilla_tubos_kjeldahl
  16: gradilla_pipetas
  17: piseta_reactivo
  18: cilindro_gas
  19: bidon_agua_destilada
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml_content.strip())

print("✅ Archivo data.yaml configurado con las 20 clases:")
!cat data.yaml

## 🚀 Paso 4: Entrenar YOLO11 Nano (`yolo11n.pt`) en GPU
Entrenamiento con 60 épocas y aumentación de datos fotométrica y espacial para máxima generalización.

In [ ]:
from ultralytics import YOLO

# Cargar modelo base YOLO11 Nano preentrenado
model = YOLO('yolo11n.pt')

# Iniciar entrenamiento de alta precisión
results = model.train(
    data='data.yaml',
    epochs=60,          # 60 épocas con early stopping
    imgsz=640,          # Tamaño de entrada para móvil TFLite
    batch=16,           # Lote óptimo para GPU T4
    patience=20,        # Paciencia de convergencia
    save=True,
    device=0,
    workers=4,
    project='yolo11_uteq_preciso',
    name='train_v1',
    exist_ok=True,
    # Aumentación de datos para laboratorio
    mosaic=1.0,
    mixup=0.15,
    degrees=10.0,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4
)

print("🎉 ¡Entrenamiento de Alta Precisión completado exitosamente!")

## 📊 Paso 5: Evaluar Rendimiento y Métricas (mAP)

In [ ]:
# Evaluar en conjunto de validación
metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

## 📱 Paso 6: Exportar YOLO11 a TensorFlow Lite (`.tflite`)

In [ ]:
import glob
import shutil
from ultralytics import YOLO

# Cargar los mejores pesos obtenidos
best_weights = glob.glob('/content/**/weights/best.pt', recursive=True)[0]
print(f"Cargando mejores pesos desde: {best_weights}")
best_model = YOLO(best_weights)

# Exportar a formato TFLite para Android (640x640)
print("Exportando a TensorFlow Lite...")
exported_path = best_model.export(format='tflite', imgsz=640)
print(f"Exportado a: {exported_path}")

# Renombrar a yolo11_bromatologia.tflite
tflite_files = glob.glob('/content/**/*.tflite', recursive=True)
target_tflite = 'yolo11_bromatologia.tflite'
shutil.copyfile(tflite_files[0], target_tflite)
print(f"✅ ¡Archivo final listo para Android!: {target_tflite} ({os.path.getsize(target_tflite)/(1024*1024):.2f} MB)")

## ⬇️ Paso 7: Descargar el Archivo `.tflite` a tu Computadora

In [ ]:
from google.colab import files

print("Descargando yolo11_bromatologia.tflite a tu computadora...")
files.download('yolo11_bromatologia.tflite')
print("✅ Guarda el archivo descargado en la carpeta de tu proyecto:")
print("   DetectorDeMaterialesLaboratorio/app/src/main/assets/")